# Notebook 09 — Model Export and Production Registration

**AI Interview Assistant · Machine Learning Pipeline, Stage 9 of 9**

---

## Purpose

Package the trained model, register it, and verify that the **running ML service
can actually load and use it**.

## The gate is respected here

Step 1 reads the Stage 8 promotion decision. If the gate denied promotion,
**nothing is registered**. The runtime then continues to serve questions by
retrieval over the labelled dataset — which is the designed fallback, not an
outage.

Overriding a failed gate at export time would make every earlier check
decorative.

## What "export" means for this project

| Artefact | Purpose |
|---|---|
| `checkpoint.pt` | model weights (this project's own, from Stage 5/7) |
| `tokenizer/` | the custom BPE vocabulary and merge table |
| `model_card.json` | what the model is, how it was built, what it cannot do |
| `manifest.json` | SHA-256 of every file, for integrity verification |
| registry entry | how the ML service discovers and activates the model |

## The model card matters

A model card that lists only capabilities is marketing. This one records the
**limitations measured in Stage 8** — including the generation pass rate and the
weakest domain — so anyone reading it knows what the model cannot do before they
rely on it.

## Outputs

- `models/<model_id>/` — the deployable package
- `model_registry.json` — updated registry
- `reports/model_export_report.json`, `reports/figures/09_*.png`

---

In [ ]:
NOTEBOOK_ID = 9

# ─────────────────────────────────────────────────────────────────────────────
# Step 0 — Environment bootstrap
#
# Locates the project workspace so this notebook runs unchanged in Google Colab,
# a local Jupyter server, or VS Code. Every later step resolves its paths from
# WORKSPACE_DIR, so nothing below depends on where the notebook was opened.
# ─────────────────────────────────────────────────────────────────────────────
import os
import sys
import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

def locate_workspace() -> Path:
    """Return the ml-service directory, whatever environment we are in."""
    # 1. Google Colab: mount Drive so checkpoints survive a runtime restart.
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        ws = Path("/content/drive/MyDrive/ai-interview-system/ml-service")
        ws.mkdir(parents=True, exist_ok=True)
        print("Environment      : Google Colab (Drive mounted)")
        return ws
    except ImportError:
        pass

    # 2. Local: walk up from the notebook until we find the ml-service root,
    #    identified by the dataset directory it must contain.
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "dataset").is_dir() and (candidate / "notebooks").is_dir():
            print("Environment      : local")
            return candidate
    print("Environment      : local (fallback to cwd)")
    return here

WORKSPACE_DIR = locate_workspace()
os.chdir(WORKSPACE_DIR)
if str(WORKSPACE_DIR) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_DIR))

# Canonical paths used across all nine notebooks.
RAW_DIR       = WORKSPACE_DIR / "dataset" / "raw"
PROCESSED_DIR = WORKSPACE_DIR / "dataset" / "processed"
QG_DIR        = PROCESSED_DIR / "question_generator"
SPLIT_DIR     = PROCESSED_DIR / "splits"
TOKENIZER_DIR = WORKSPACE_DIR / "tokenizer"
CKPT_DIR      = WORKSPACE_DIR / "checkpoints"
MODEL_DIR     = WORKSPACE_DIR / "models"
REPORTS_DIR   = WORKSPACE_DIR / "reports"
FIGURES_DIR   = REPORTS_DIR / "figures"

for d in (RAW_DIR, PROCESSED_DIR, SPLIT_DIR, TOKENIZER_DIR, CKPT_DIR,
          MODEL_DIR, REPORTS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Workspace        : {WORKSPACE_DIR}")
print(f"Python           : {sys.version.split()[0]}")
print(f"Run started      : {datetime.now(timezone.utc).isoformat(timespec='seconds')}")

---

## Step 0b — Figure and statistics conventions

One style definition serves every figure in the nine-notebook pipeline, so
charts are directly comparable when placed side by side in the write-up.

Three conventions are fixed here:

1. **A colour-blind-safe categorical palette** — the same six colours, in the
   same order, wherever a chart encodes categories.
2. **Automatic figure export** — `save_figure()` writes every figure to
   `reports/figures/` at 200 dpi with a numbered filename, and prints its
   caption, so figures can be cited as *Figure N.k* in the dissertation.
3. **A single summary-statistics function** — `describe_series()` reports
   n, mean, sd, the five-number summary, skewness and kurtosis in a fixed
   order for every variable, so distributions are described consistently.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 0b — Plotting conventions
#
# One style definition for every figure in the pipeline, so figures across the
# nine notebooks are directly comparable in the dissertation. Every figure is
# also saved to reports/figures/ at 200 dpi, ready to drop into the write-up.
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.edgecolor": "#444444",
    "grid.alpha": 0.3,
    "legend.frameon": True,
    "figure.autolayout": False,
})

# Colour-blind-safe categorical palette, used consistently for every chart.
PALETTE = ["#3B6FD4", "#E1893B", "#3EA37A", "#C4576B", "#7B5EA7", "#8C7B68"]
sns.set_palette(PALETTE)

_figure_index = {"n": 0}

def save_figure(fig, slug: str, caption: str = "") -> Path:
    """Save a figure with a numbered filename and print its caption."""
    _figure_index["n"] += 1
    n = _figure_index["n"]
    path = FIGURES_DIR / f"{NOTEBOOK_ID:02d}_fig{n:02d}_{slug}.png"
    fig.savefig(path)
    label = f"Figure {NOTEBOOK_ID}.{n}"
    if caption:
        print(f"{label}: {caption}")
    print(f"           saved -> {path.relative_to(WORKSPACE_DIR)}")
    return path

def describe_series(series: pd.Series, name: str) -> pd.Series:
    """Summary statistics reported in a consistent order for every variable."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    return pd.Series({
        "n": len(s),
        "mean": s.mean(),
        "std": s.std(ddof=1),
        "min": s.min(),
        "q1": s.quantile(0.25),
        "median": s.median(),
        "q3": s.quantile(0.75),
        "max": s.max(),
        "skew": s.skew(),
        "kurtosis": s.kurtosis(),
    }, name=name)

print("Plot style       : configured")
print(f"Figure output    : {FIGURES_DIR.relative_to(WORKSPACE_DIR)}")
print(f"Palette          : {len(PALETTE)} colour-blind-safe categories")

---

## Step 1 — Read the Stage 8 promotion decision

The gate is read first, before any packaging work, so a denied model consumes no
further effort and — more importantly — cannot be registered by accident.

In [ ]:
import shutil
import hashlib
import torch

from transformer_scratch import CustomBPETokenizer, load_checkpoint
from model_registry import registry

EVAL_FILE = REPORTS_DIR / "fine_tuned_model_evaluation.json"
assert EVAL_FILE.exists(), (
    f"{EVAL_FILE.name} missing — run Notebook 08 first. A model cannot be "
    f"registered without a held-out evaluation."
)

evaluation = json.loads(EVAL_FILE.read_text(encoding="utf-8"))
gate = evaluation["promotion_gate"]
PROMOTION_APPROVED = bool(gate["approved"])

print("STAGE 8 PROMOTION GATE")
print("=" * 84)
for criterion in gate["criteria"]:
    status = "PASS" if criterion["passed"] else "FAIL"
    print(f"  [{status}] {criterion['description']}")
    print(f"         measured {criterion['value']:.4f}  "
          f"threshold {criterion['threshold']:.4f}")
print("=" * 84)
print(f"  Verdict: {'APPROVED' if PROMOTION_APPROVED else 'DENIED'}")
if not PROMOTION_APPROVED:
    print(f"  Failed  : {gate['failed_criteria']}")
print("=" * 84)

if not PROMOTION_APPROVED:
    print("\n" + "!" * 84)
    print("REGISTRATION BLOCKED")
    print("!" * 84)
    print("The model did not pass the held-out promotion gate, so it will not")
    print("be registered as active. Overriding this decision would make every")
    print("earlier verification stage decorative.")
    print()
    print("Runtime behaviour while no model is registered:")
    print("  The ML service's question planner retrieves from the labelled")
    print("  question dataset (`interview_engine.QuestionPool`), which is the")
    print("  designed fallback. Interviews continue to work normally.")
    print()
    print("The steps below still run, packaging the artefacts and writing the")
    print("model card, but the registry status is set to 'rejected' rather than")
    print("'production', and the model is NOT activated.")
    print("!" * 84)

test_metrics = evaluation["test_metrics"]
model_info = evaluation["model"]
print(f"\nModel under consideration : {model_info['architecture']} "
      f"({model_info['kind']})")
print(f"Test loss                 : {test_metrics['loss']:.4f}")
print(f"Test perplexity           : {test_metrics['perplexity']:.2f}")

---

## Step 2 — Assign a semantic version

Versioning follows semantic versioning, driven by what changed:

- **MAJOR** — the architecture or the tokenizer changed, so the model is not
  interchangeable with the previous one.
- **MINOR** — retrained or specialised on the same architecture.
- **PATCH** — packaging or metadata only, weights unchanged.

The version is derived from what is already in the registry, so re-running this
notebook does not silently overwrite a registered model.

In [ ]:
CAPABILITY = "question_generator"
MODEL_FAMILY = "ai-interview-question-generator"

existing = registry.list_models(CAPABILITY) or {}
print(f"Models already registered under '{CAPABILITY}': {len(existing)}")
for model_id, record in existing.items():
    print(f"  {model_id:52s} status={record.get('status')}")

def next_version(existing_models: dict, architecture: str,
                 tokenizer_vocab: int) -> str:
    """Bump MAJOR on an architecture/tokenizer change, MINOR otherwise."""
    versions = []
    for model_id, record in existing_models.items():
        match = re.search(r"v(\d+)\.(\d+)\.(\d+)$", model_id)
        if match:
            versions.append((tuple(int(g) for g in match.groups()), record))
    if not versions:
        return "1.0.0"

    versions.sort(key=lambda x: x[0])
    (major, minor, patch), latest = versions[-1]

    architecture_changed = latest.get("architecture") != architecture
    tokenizer_changed = latest.get("tokenizer_vocab_size") != tokenizer_vocab
    if architecture_changed or tokenizer_changed:
        return f"{major + 1}.0.0"
    return f"{major}.{minor + 1}.0"

import re

tokenizer = CustomBPETokenizer.load(TOKENIZER_DIR)
TOKENIZER_VOCAB = int(getattr(tokenizer, "vocab_size", 0) or
                      len(getattr(tokenizer, "vocab", {})) or 4096)

VERSION = next_version(existing, model_info["candidate_id"], TOKENIZER_VOCAB)
MODEL_ID = f"{MODEL_FAMILY}-v{VERSION}"

print(f"\nVERSION ASSIGNMENT")
print("=" * 74)
print(f"  Architecture     : {model_info['candidate_id']}")
print(f"  Tokenizer vocab  : {TOKENIZER_VOCAB:,}")
print(f"  Assigned version : {VERSION}")
print(f"  Model id         : {MODEL_ID}")
print("=" * 74)

---

## Step 3 — Package the artefacts

Weights and tokenizer are copied into `models/<model_id>/`, and every file is
hashed. The manifest lets the ML service verify at load time that the package
has not been corrupted or partially written.

In [ ]:
SOURCE_CKPT = WORKSPACE_DIR / model_info["checkpoint"]
assert SOURCE_CKPT.exists(), f"Checkpoint {SOURCE_CKPT} not found."

PACKAGE_DIR = MODEL_DIR / MODEL_ID
if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

# 1. Weights.
for item in SOURCE_CKPT.iterdir():
    if item.is_file():
        shutil.copy2(item, PACKAGE_DIR / item.name)

# 2. Tokenizer, so the package is self-contained.
package_tokenizer_dir = PACKAGE_DIR / "tokenizer"
package_tokenizer_dir.mkdir(exist_ok=True)
for item in TOKENIZER_DIR.iterdir():
    if item.is_file():
        shutil.copy2(item, package_tokenizer_dir / item.name)

# 3. Manifest with a hash per file.
def file_digest(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

manifest_files = {}
total_bytes = 0
for path in sorted(PACKAGE_DIR.rglob("*")):
    if path.is_file() and path.name != "manifest.json":
        relative = path.relative_to(PACKAGE_DIR).as_posix()
        size = path.stat().st_size
        manifest_files[relative] = {"bytes": size, "sha256": file_digest(path)}
        total_bytes += size

manifest = {
    "model_id": MODEL_ID,
    "version": VERSION,
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "file_count": len(manifest_files),
    "total_bytes": total_bytes,
    "files": manifest_files,
}
(PACKAGE_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2),
                                           encoding="utf-8")

print("PACKAGE CONTENTS")
print("=" * 84)
for relative, meta in manifest_files.items():
    print(f"  {relative:38s} {meta['bytes'] / 1024**2:8.2f} MB  "
          f"{meta['sha256'][:16]}...")
print("-" * 84)
print(f"  {'TOTAL':38s} {total_bytes / 1024**2:8.2f} MB  "
      f"({len(manifest_files)} files)")
print("=" * 84)
print(f"\nPackage: {PACKAGE_DIR.relative_to(WORKSPACE_DIR)}")

---

## Step 4 — Write the model card

The card records what the model is, how it was produced, **and what it cannot
do**. The limitations section is populated from Stage 8's measurements, not
written from optimism.

In [ ]:
training_report = json.loads(
    (REPORTS_DIR / "candidate_training_report.json").read_text(encoding="utf-8"))
selection = json.loads(
    (REPORTS_DIR / "model_selection.json").read_text(encoding="utf-8"))
split_report = json.loads(
    (REPORTS_DIR / "split_report.json").read_text(encoding="utf-8"))
dataset_metadata = json.loads(
    (REPORTS_DIR / "dataset_metadata.json").read_text(encoding="utf-8"))

weakest = evaluation.get("weakest_domain", {})
generation = evaluation.get("generation", {})

model_card = {
    "model_id": MODEL_ID,
    "version": VERSION,
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "capability": CAPABILITY,
    "task": "Generate a technical interview question for a given domain "
            "and difficulty.",

    "provenance": {
        "trained_from_scratch": True,
        "pretrained_weights_used": False,
        "adapters_or_lora_used": False,
        "external_llm_used": False,
        "statement": (
            "Every parameter in this model was randomly initialised and trained "
            "within this project. Hugging Face was used only as a dataset host. "
            "Notebook 01 includes an audit that fails if any pretrained weight "
            "file is present in the workspace."
        ),
    },

    "architecture": {
        "family": "decoder-only Transformer (causal language model)",
        "candidate_id": model_info["candidate_id"],
        "label": model_info["architecture"],
        "parameters": model_info["parameters"],
        "selected_because": selection["selected"]["hypothesis"],
        "selection_score": selection["selected"]["composite_score"],
        "selection_robust_to_weighting": selection["sensitivity"]["robust"],
    },

    "tokenizer": {
        "type": "custom byte-pair encoding, trained in-project",
        "vocab_size": TOKENIZER_VOCAB,
        "fitted_on": "training split only (no validation or test text)",
    },

    "training_data": {
        "source": dataset_metadata["dataset"]["identifier"],
        "source_url": dataset_metadata["dataset"]["url"],
        "licence": dataset_metadata["dataset"]["licence"],
        "raw_records": dataset_metadata["dataset"]["record_count"],
        "after_cleaning": split_report["sizes"]["corpus"],
        "train": split_report["sizes"]["train"],
        "validation": split_report["sizes"]["validation"],
        "test": split_report["sizes"]["test"],
        "split_seed": split_report["seed"],
        "stratified_on": split_report["stratification_key"],
        "leakage_verified_zero": split_report["leakage"]["zero_exact_leakage"],
    },

    "evaluation": {
        "held_out_test_loss": test_metrics["loss"],
        "held_out_test_perplexity": test_metrics["perplexity"],
        "top1_next_token_accuracy_pct": test_metrics["top1_accuracy"],
        "top5_next_token_accuracy_pct": test_metrics["top5_accuracy"],
        "validation_test_gap": evaluation["val_test_gap"],
        "test_split_read_once": True,
        "promotion_gate_passed": PROMOTION_APPROVED,
    },

    # The section that makes this card useful rather than promotional.
    "limitations": {
        "measured": evaluation.get("limitations", []),
        "generation_pass_rate": generation.get("pass_rate"),
        "weakest_domain": weakest.get("domain"),
        "weakest_domain_loss": weakest.get("mean_loss"),
        "guidance": (
            "This model is one of two question sources in the runtime. The ML "
            "service gates generated questions on a quality check "
            "(interview_engine.QuestionPool.quality_score) and falls back to "
            "retrieval over the labelled dataset when a generated question "
            "fails. Do not deploy this model as the sole question source."
        ),
    },

    "intended_use": {
        "in_scope": [
            "Generating candidate interview questions inside this system, "
            "behind the runtime quality gate.",
            "Academic demonstration of an end-to-end from-scratch ML pipeline.",
        ],
        "out_of_scope": [
            "Any hiring or selection decision about a real candidate.",
            "Use as a general-purpose language model.",
            "Deployment as the only source of interview questions.",
        ],
    },

    "reproducibility": {
        "seed": training_report["reproducibility"]["seed"],
        "torch_version": training_report["reproducibility"]["torch_version"],
        "device_trained_on": training_report["reproducibility"]["device"],
        "pipeline": "Notebooks 01-09 in ml-service/notebooks/, run in order.",
    },
}

card_path = PACKAGE_DIR / "model_card.json"
card_path.write_text(json.dumps(model_card, indent=2, default=str),
                     encoding="utf-8")

print("MODEL CARD WRITTEN")
print("=" * 78)
print(f"  Path : {card_path.relative_to(WORKSPACE_DIR)}")
print("\n  Declared limitations:")
for limitation in model_card["limitations"]["measured"]:
    print(f"    - {limitation}")
print(f"\n  Generation pass rate : "
      f"{model_card['limitations']['generation_pass_rate']}")
print(f"  Weakest domain       : {model_card['limitations']['weakest_domain']} "
      f"(loss {model_card['limitations']['weakest_domain_loss']})")
print("\n  Out of scope:")
for item in model_card["intended_use"]["out_of_scope"]:
    print(f"    - {item}")
print("=" * 78)

---

## Step 5 — Register in the model registry

The registry is what the ML service reads at startup to discover which model to
load. Status reflects the gate:

- gate **approved** → `production`, and the model is activated
- gate **denied** → `rejected`, and no activation occurs

Recording a rejected model rather than discarding it keeps the negative result
in the audit trail, which is exactly what a research pipeline should preserve.

In [ ]:
STATUS = "production" if PROMOTION_APPROVED else "rejected"

record = {
    "model_id": MODEL_ID,
    "capability": CAPABILITY,
    "version": VERSION,
    "status": STATUS,
    "model_type": "scratch_trained",
    "architecture": model_info["candidate_id"],
    "architecture_label": model_info["architecture"],
    "parameters": model_info["parameters"],
    "parameters_display": f"{model_info['parameters'] / 1e6:.1f}M "
                          f"(own architecture)",
    "tokenizer": "custom_bpe",
    "tokenizer_vocab_size": TOKENIZER_VOCAB,
    "storage_path": str(PACKAGE_DIR.relative_to(WORKSPACE_DIR)).replace("\\", "/"),
    "manifest_sha256": hashlib.sha256(
        (PACKAGE_DIR / "manifest.json").read_bytes()).hexdigest(),
    "metrics": {
        "test_loss": round(test_metrics["loss"], 5),
        "test_perplexity": round(test_metrics["perplexity"], 3),
        "top1_accuracy_pct": round(test_metrics["top1_accuracy"], 3),
        "top5_accuracy_pct": round(test_metrics["top5_accuracy"], 3),
        "generation_pass_rate": generation.get("pass_rate"),
    },
    "promotion_gate": {
        "approved": PROMOTION_APPROVED,
        "failed_criteria": gate["failed_criteria"],
    },
    "pretrained_weights_used": False,
    "model_card": "model_card.json",
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}

registered = registry.register_model(record)
print(f"Registered: {registered['model_id']}  status={registered['status']}")

if PROMOTION_APPROVED:
    activated = registry.set_active_model(CAPABILITY, MODEL_ID)
    print(f"Activated as the live {CAPABILITY}.")
else:
    print(f"NOT activated — the promotion gate denied this model.")
    current = registry.get_active_model(CAPABILITY)
    if current:
        print(f"The active model remains: {current.get('model_id')}")
    else:
        print("No model is active. The runtime uses dataset retrieval, which")
        print("is the designed fallback.")

print("\nREGISTRY STATE")
print("=" * 84)
for model_id, entry in (registry.list_models(CAPABILITY) or {}).items():
    active = registry.get_active_model(CAPABILITY) or {}
    marker = "  <- ACTIVE" if active.get("model_id") == model_id else ""
    print(f"  {model_id:48s} {entry.get('status'):12s}{marker}")
print("=" * 84)

---

## Step 6 — Verify the ML service can actually load it

The step that catches the failure a hash check cannot: the package exists and is
intact, but the running service cannot load it or produces nothing.

The verification loads the package **the same way the ML service does** and runs
a real generation, then puts the output through the runtime's own quality gate.

In [ ]:
verification = []

def record_check(name, passed, detail=""):
    verification.append({"check": name, "passed": bool(passed),
                         "detail": str(detail)})
    print(f"  [{'PASS' if passed else 'FAIL'}] {name}"
          f"{'  — ' + str(detail) if detail else ''}")

print("DEPLOYMENT VERIFICATION")
print("=" * 84)

# 1. Manifest integrity.
try:
    stored = json.loads((PACKAGE_DIR / "manifest.json").read_text(encoding="utf-8"))
    mismatches = [
        relative for relative, meta in stored["files"].items()
        if file_digest(PACKAGE_DIR / relative) != meta["sha256"]
    ]
    record_check("manifest integrity", not mismatches,
                 f"{len(stored['files'])} files verified"
                 if not mismatches else f"mismatched: {mismatches}")
except Exception as exc:
    record_check("manifest integrity", False, exc)

# 2. Load the package exactly as the service does.
loaded_model = None
try:
    loaded_model, loaded_payload = load_checkpoint(PACKAGE_DIR, device="cpu")
    loaded_model.eval()
    record_check("checkpoint loads from the package", True,
                 f"{loaded_model.count_parameters():,} parameters")
except Exception as exc:
    record_check("checkpoint loads from the package", False, exc)

# 3. Tokenizer loads from the package (self-containment).
package_tokenizer = None
try:
    package_tokenizer = CustomBPETokenizer.load(PACKAGE_DIR / "tokenizer")
    round_trip = package_tokenizer.decode(
        package_tokenizer.encode("What is a database index?"),
        skip_special_tokens=True)
    record_check("tokenizer loads and round-trips", bool(round_trip.strip()),
                 f"decoded: {round_trip[:44]!r}")
except Exception as exc:
    record_check("tokenizer loads and round-trips", False, exc)

# 4. A real forward pass produces sane logits.
if loaded_model is not None and package_tokenizer is not None:
    try:
        ids = package_tokenizer.encode("<DOMAIN: SQL> <DIFFICULTY: Beginner>")
        tensor = torch.tensor([ids], dtype=torch.long)
        with torch.no_grad():
            logits = loaded_model(tensor)
            if isinstance(logits, tuple):
                logits = logits[0]
        finite = bool(torch.isfinite(logits).all())
        record_check("forward pass produces finite logits", finite,
                     f"shape {tuple(logits.shape)}")
    except Exception as exc:
        record_check("forward pass produces finite logits", False, exc)

    # 5. Generation, then the runtime's own quality gate.
    try:
        with torch.no_grad():
            output = loaded_model.generate(tensor, max_new_tokens=26,
                                           temperature=0.8, top_k=40)
        text = package_tokenizer.decode(output[0].tolist(),
                                        skip_special_tokens=True)
        for token in ("<DOMAIN:", "<DIFFICULTY:", ">", "SQL", "Beginner"):
            text = text.replace(token, " ")
        text = " ".join(text.split()).strip()
        record_check("generation produces output", bool(text),
                     f"{text[:52]!r}")

        # The gate the ML service applies before speaking a question aloud.
        try:
            sys.path.insert(0, str(WORKSPACE_DIR))
            from interview_engine import QuestionPool
            score = QuestionPool.quality_score(text)
            gate_pass = score >= QuestionPool.MIN_QUALITY
            record_check("runtime quality gate evaluated", True,
                         f"score {score:.2f} vs threshold "
                         f"{QuestionPool.MIN_QUALITY} -> "
                         f"{'would be spoken' if gate_pass else 'would fall back to retrieval'}")
        except Exception as exc:
            record_check("runtime quality gate evaluated", False, exc)
    except Exception as exc:
        record_check("generation produces output", False, exc)

# 6. The registry entry resolves to the package on disk.
try:
    active = registry.get_active_model(CAPABILITY)
    if PROMOTION_APPROVED:
        resolved = WORKSPACE_DIR / (active or {}).get("storage_path", "")
        record_check("registry path resolves to the package",
                     resolved.exists() and resolved.samefile(PACKAGE_DIR),
                     str(resolved.relative_to(WORKSPACE_DIR))
                     if resolved.exists() else "not found")
    else:
        record_check("registry correctly withheld activation",
                     (active or {}).get("model_id") != MODEL_ID,
                     "rejected model was not activated")
except Exception as exc:
    record_check("registry path resolves to the package", False, exc)

print("=" * 84)
passed = sum(v["passed"] for v in verification)
DEPLOYMENT_READY = passed == len(verification)
print(f"  {passed} of {len(verification)} verification checks passed")
print("=" * 84)

In [ ]:
# ── Figure 9.1 — pipeline summary ──────────────────────────────────────────
fig = plt.figure(figsize=(16.5, 9))
grid = fig.add_gridspec(2, 3, hspace=0.42, wspace=0.28)

# Panel 1: corpus attrition through the pipeline.
ax = fig.add_subplot(grid[0, 0])
stages = ["Raw", "Cleaned", "Train", "Val", "Test"]
sizes = [
    dataset_metadata["dataset"]["record_count"],
    split_report["sizes"]["corpus"],
    split_report["sizes"]["train"],
    split_report["sizes"]["validation"],
    split_report["sizes"]["test"],
]
bars = ax.bar(stages, sizes, color=[PALETTE[0], PALETTE[1], PALETTE[2],
                                    PALETTE[3], PALETTE[4]])
ax.set_title("Corpus through the pipeline")
ax.set_ylabel("Records")
ax.bar_label(bars, fmt="%d", padding=2, fontsize=8)
ax.tick_params(axis="x", labelsize=8)
ax.margins(y=0.18)

# Panel 2: candidate validation losses, winner highlighted.
ax = fig.add_subplot(grid[0, 1])
scorecard = pd.DataFrame(selection["scorecard"])
colours = [PALETTE[2] if r == 1 else PALETTE[0] for r in scorecard["rank"]]
bars = ax.barh(scorecard["label"][::-1], scorecard["val_loss"][::-1],
               color=colours[::-1])
ax.set_title("Candidate validation loss")
ax.set_xlabel("Validation cross-entropy")
ax.tick_params(axis="y", labelsize=7.5)
ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=7.5)
ax.margins(x=0.2)

# Panel 3: test metrics.
ax = fig.add_subplot(grid[0, 2])
metric_names = ["Top-1\nacc %", "Top-5\nacc %", "Gen pass\nrate %"]
metric_values = [test_metrics["top1_accuracy"], test_metrics["top5_accuracy"],
                 (generation.get("pass_rate") or 0) * 100]
bars = ax.bar(metric_names, metric_values, color=PALETTE[:3])
ax.set_title("Held-out test performance")
ax.set_ylabel("Percent")
ax.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=8.5)
ax.tick_params(axis="x", labelsize=8)
ax.margins(y=0.2)

# Panel 4: promotion gate.
ax = fig.add_subplot(grid[1, 0])
gate_names = [c["name"].replace("_", "\n")[:22] for c in gate["criteria"]]
gate_passed = [1 if c["passed"] else 0 for c in gate["criteria"]]
bars = ax.barh(gate_names[::-1], [1] * len(gate_names),
               color=[PALETTE[2] if p else PALETTE[3]
                      for p in gate_passed][::-1])
ax.set_title(f"Promotion gate — {'APPROVED' if PROMOTION_APPROVED else 'DENIED'}")
ax.set_xlim(0, 1.25)
ax.set_xticks([])
ax.tick_params(axis="y", labelsize=7)
for i, p in enumerate(gate_passed[::-1]):
    ax.annotate("PASS" if p else "FAIL", xy=(1.03, i), va="center",
                fontsize=7.5, fontweight="bold",
                color=PALETTE[2] if p else PALETTE[3])

# Panel 5: deployment verification.
ax = fig.add_subplot(grid[1, 1])
check_names = [v["check"][:26] for v in verification]
bars = ax.barh(check_names[::-1], [1] * len(verification),
               color=[PALETTE[2] if v["passed"] else PALETTE[3]
                      for v in verification][::-1])
ax.set_title(f"Deployment verification — {passed}/{len(verification)}")
ax.set_xlim(0, 1.05)
ax.set_xticks([])
ax.tick_params(axis="y", labelsize=7)

# Panel 6: the release summary.
ax = fig.add_subplot(grid[1, 2])
ax.axis("off")
status_colour = PALETTE[2] if (PROMOTION_APPROVED and DEPLOYMENT_READY) else PALETTE[3]
headline = ("REGISTERED\n& ACTIVE" if (PROMOTION_APPROVED and DEPLOYMENT_READY)
            else "NOT ACTIVATED")
ax.text(0.5, 0.82, headline, ha="center", va="center", fontsize=22,
        fontweight="bold", color=status_colour, transform=ax.transAxes)
lines = [
    f"model  : {MODEL_ID[-34:]}",
    f"version: {VERSION}",
    f"status : {STATUS}",
    f"params : {model_info['parameters'] / 1e6:.2f}M",
    f"size   : {total_bytes / 1024**2:.1f} MB",
    f"loss   : {test_metrics['loss']:.4f}",
    f"ppl    : {test_metrics['perplexity']:.2f}",
    "pretrained weights: NONE",
]
ax.text(0.5, 0.36, "\n".join(lines), ha="center", va="center", fontsize=9,
        family="monospace", transform=ax.transAxes,
        bbox=dict(boxstyle="round,pad=0.7", fc="#F5F7FA", ec="#CCCCCC"))

fig.suptitle("AI Interview Assistant — nine-stage ML pipeline, final summary",
             y=0.97, fontsize=15, fontweight="bold")
save_figure(fig, "pipeline_summary",
            "The complete pipeline on one page: corpus attrition, architecture "
            "selection, held-out performance, the promotion gate, and "
            "deployment verification.")
plt.show()

---

## Step 7 — Export report

In [ ]:
export_report = {
    "stage": "09_model_export_and_registration",
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "promotion_gate_approved": PROMOTION_APPROVED,
    "model": {
        "model_id": MODEL_ID,
        "version": VERSION,
        "status": STATUS,
        "activated": bool(PROMOTION_APPROVED),
        "architecture": model_info["candidate_id"],
        "parameters": model_info["parameters"],
    },
    "package": {
        "path": str(PACKAGE_DIR.relative_to(WORKSPACE_DIR)),
        "file_count": len(manifest_files),
        "total_bytes": total_bytes,
        "total_mb": round(total_bytes / 1024**2, 2),
        "manifest_sha256": record["manifest_sha256"],
    },
    "model_card": str(card_path.relative_to(WORKSPACE_DIR)),
    "registry": {
        "capability": CAPABILITY,
        "registered": True,
        "active_model": (registry.get_active_model(CAPABILITY) or {}).get("model_id"),
    },
    "verification": verification,
    "verification_passed": f"{passed}/{len(verification)}",
    "deployment_ready": bool(DEPLOYMENT_READY and PROMOTION_APPROVED),
    "runtime_note": (
        "The ML service reads model_registry.json at startup. When no model is "
        "active, interview_engine.QuestionPool serves questions by retrieval "
        "over the labelled dataset, so interviews function either way."
    ),
}

report_path = REPORTS_DIR / "model_export_report.json"
report_path.write_text(json.dumps(export_report, indent=2, default=str),
                       encoding="utf-8")

print("=" * 84)
print("PIPELINE COMPLETE — STAGE 9 OF 9")
print("=" * 84)
print(f"  Model id            : {MODEL_ID}")
print(f"  Status              : {STATUS}")
print(f"  Activated           : {PROMOTION_APPROVED}")
print(f"  Package             : {PACKAGE_DIR.relative_to(WORKSPACE_DIR)} "
      f"({total_bytes / 1024**2:.1f} MB)")
print(f"  Verification        : {passed}/{len(verification)} checks passed")
print(f"  Export report       : {report_path.relative_to(WORKSPACE_DIR)}")
print(f"  Total figures       : "
      f"{len(sorted(FIGURES_DIR.glob('*.png')))} across all nine notebooks")
print("=" * 84)

---

## Stage 9 summary

| Step | Outcome |
|---|---|
| Promotion gate honoured | a denied model is recorded as `rejected`, never activated |
| Semantic version assigned | MAJOR on architecture/tokenizer change, MINOR otherwise |
| Package built | weights + tokenizer + card + per-file SHA-256 manifest |
| Model card written | includes the limitations **measured** in Stage 8 |
| Registry updated | the ML service discovers the model from here |
| Deployment verified | loaded, forward pass, generation, runtime quality gate |

---

# The complete pipeline

| Stage | Notebook | Contribution |
|---|---|---|
| 1 | Dataset acquisition | provenance, SHA-256, zero-pretrained-weights audit |
| 2 | Exploratory analysis | 9 figures; every cleaning threshold justified |
| 3 | Preprocessing | 8 rules applied, each with before/after verification |
| 4 | Validation & splitting | stratified 80/10/10, leakage tested, test split sealed |
| 5 | Tokenizer & training | custom BPE + 4 from-scratch architectures |
| 6 | Comparison & selection | multi-criteria, weights declared first, sensitivity tested |
| 7 | Specialisation | continued training at LR/5, forgetting checked |
| 8 | Held-out evaluation | single authorised test read, 5-criterion promotion gate |
| 9 | Export & registration | packaged, carded, registered, deployment verified |

## What makes this pipeline defensible

1. **No pretrained weights.** Audited in Stage 1, asserted again in Stage 7.
2. **Every threshold is justified by a measurement.** Stage 3 reads its limits
   from Stage 2's report rather than hard-coding them.
3. **The test split was read once**, by the only notebook authorised to do so,
   with its hash verified against the Stage 4 seal.
4. **Decisions were made before the data that could bias them was seen** —
   selection weights before scores, gate thresholds before the test read.
5. **Negative results are kept.** A denied model is registered as `rejected`
   rather than deleted, and the weakest domain is named rather than averaged
   away.
6. **The known limitation is disclosed and mitigated.** Generation fluency is
   limited by corpus size; the model card says so, and the runtime falls back to
   retrieval.